# Học Machine Learning với dữ liệu điểm học sinh

Notebook này dạy từng bước: linear regression, logistic regression, cây quyết định và random forest. Mỗi bước có phần giải thích, việc cần làm và kết quả mong đợi để bạn tự kiểm tra.

Chuẩn bị:
1. Đặt file data1.csv cùng thư mục với notebook.
2. Cài thư viện nếu chưa có: pip install pandas numpy matplotlib scikit-learn statsmodels

Cách làm: đọc phần giải thích, viết code vào ô trống, chạy bằng Shift + Enter, rồi so sánh với kết quả mong đợi. Nếu kết quả khác, đọc lại đề bài và thử lại trước khi xem đáp án.

## Bước 0: Import thư viện

Mục tiêu: nạp các thư viện dùng trong toàn bộ notebook. Ô này đã viết sẵn, bạn chỉ cần chạy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

## Bước 1: Đọc dữ liệu

Mục tiêu: đưa file CSV vào bộ nhớ dưới dạng DataFrame của pandas, rồi kiểm tra kích thước.

Kiến thức: pd.read_csv đọc file thành bảng. df.shape trả về (số dòng, số cột).

Kết quả mong đợi: (649, 33).

In [ ]:
df = pd.read_csv('data1.csv')
print(df.shape)

## Bước 2: Xem tổng quan

Mục tiêu: xem vài dòng đầu, kiểu dữ liệu của từng cột, và kiểm tra giá trị thiếu.

Kiến thức: head() xem các dòng đầu, info() cho biết kiểu dữ liệu, isna().sum() đếm ô trống.

Kết quả mong đợi: 17 cột chuỗi, 16 cột số, 0 ô trống.

Câu hỏi: tại sao phải kiểm tra giá trị thiếu trước khi huấn luyện mô hình?

In [ ]:
print(df.head())
df.info()
print('Số ô bị thiếu:', df.isna().sum().sum())

## Bước 3: Phân phối điểm G3

Mục tiêu: xem G3 phân bố thế nào và có điểm bất thường không.

Kiến thức: value_counts() đếm số lần xuất hiện của mỗi giá trị. plt.hist vẽ biểu đồ phân phối.

Kết quả mong đợi: 15 học sinh có G3 = 0. Phần lớn điểm nằm từ 8 đến 15.

Câu hỏi: điểm 0 có nghĩa gì trong thực tế? Bạn có nên giữ những dòng này không?

In [ ]:
print(df['G3'].value_counts().sort_index())
print('Số học sinh G3 = 0:', (df['G3'] == 0).sum())

plt.hist(df['G3'], bins=20)
plt.xlabel('G3')
plt.ylabel('Số học sinh')
plt.title('Phân phối điểm cuối kỳ G3')
plt.show()

## Bước 4: Tương quan với G3

Mục tiêu: xem biến nào có liên hệ tuyến tính mạnh với G3.

Kiến thức: corr() tính hệ số tương quan Pearson giữa các cột số, giá trị từ -1 đến 1. select_dtypes('number') chọn các cột số.

Kết quả mong đợi: G2 khoảng 0,919, G1 khoảng 0,826.

Câu hỏi: tại sao G2 gần như trùng với G3? Điều này ảnh hưởng thế nào đến việc đánh giá mô hình?

In [ ]:
corr_g3 = df.select_dtypes('number').corr()['G3']
print(corr_g3.sort_values(ascending=False).round(3))

## Bước 5: Mã hóa các cột có hai giá trị

Mục tiêu: đổi các cột chữ có hai giá trị (yes/no, F/M) thành số 0 và 1. Hồi quy chỉ làm việc với số.

Kiến thức: astype('category').cat.codes gán mã số theo thứ tự chữ cái. Ví dụ 'no' thành 0 và 'yes' thành 1, 'F' thành 0 và 'M' thành 1.

Kết quả mong đợi: mỗi cột trong binary_cols chỉ còn giá trị 0 hoặc 1.

In [ ]:
binary_cols = ['sex', 'schoolsup', 'famsup', 'paid', 'higher', 'internet', 'romantic']
for c in binary_cols:
    df[c] = df[c].astype('category').cat.codes
print(df[binary_cols].head())

## Bước 6: Chọn feature và tạo biến mục tiêu

Mục tiêu: tạo hai bộ feature (setup A và setup B) và tách biến mục tiêu y.

Kiến thức: feature là danh sách tên cột. df[danh_sách] lấy ra nhiều cột cùng lúc. Setup A không có G1 và G2. Setup B thêm hai cột này vào.

Kết quả mong đợi: len(feat_A) = 20, len(feat_B) = 22.

Câu hỏi: nếu dùng setup B để dự đoán điểm cho học sinh chưa thi kỳ nào, có được không? Vì sao?

In [ ]:
feat_A = ['sex', 'age', 'Medu', 'Fedu', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid',
          'higher', 'internet', 'romantic', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'traveltime', 'freetime', 'famrel']
feat_B = feat_A + ['G1', 'G2']
X_A = df[feat_A]
X_B = df[feat_B]
y = df['G3']
print(len(feat_A), len(feat_B))

## Bước 7: Chia tập train và test

Mục tiêu: giữ lại 20% dữ liệu để kiểm tra mô hình trên dữ liệu chưa thấy.

Kiến thức: train_test_split chia ngẫu nhiên. random_state=42 giữ kết quả cố định mỗi lần chạy. Mô hình chỉ học trên tập train. Kết quả trên tập test mới cho biết khả năng dự đoán thật.

Kết quả mong đợi: X_test_A có 130 dòng và 20 cột.

In [ ]:
X_train_A, X_test_A, y_train, y_test = train_test_split(X_A, y, test_size=0.2, random_state=42)
X_train_B, X_test_B, _, _ = train_test_split(X_B, y, test_size=0.2, random_state=42)
print(X_test_A.shape)

## Bước 8: Linear regression với sklearn

Mục tiêu: huấn luyện hồi quy tuyến tính và đo độ chính xác trên tập test.

Kiến thức: quy trình của sklearn có ba bước: tạo mô hình, fit trên tập train, predict trên tập test. R² đo tỷ lệ phương sai được giải thích, 1 là hoàn hảo, dưới 0 là tệ hơn việc đoán trung bình. MAE là sai số trung bình tuyệt đối, tính theo đơn vị điểm.

Kết quả mong đợi: R² test khoảng 0,219. MAE khoảng 2,07.

Câu hỏi: nếu luôn đoán bằng điểm trung bình của tập train, MAE là bao nhiêu? So sánh với 2,07.

In [ ]:
lr = LinearRegression()
lr.fit(X_train_A, y_train)
pred = lr.predict(X_test_A)
print('R² test:', round(r2_score(y_test, pred), 3))
print('MAE test:', round(mean_absolute_error(y_test, pred), 2))

baseline_pred = np.full(len(y_test), y_train.mean())
print('MAE nếu đoán trung bình:', round(mean_absolute_error(y_test, baseline_pred), 2))

## Bước 9: Đọc bảng thống kê với statsmodels

Mục tiêu: đọc hệ số, p-value và khoảng tin cậy. sklearn không đưa các thông tin này, nên dùng statsmodels cho phần suy luận.

Kiến thức: sm.add_constant thêm cột hằng số (intercept). sm.OLS là bình phương tối thiểu thông thường, tương đương linear regression. Cột P>|t| trong bảng là p-value. Cột coef là thay đổi trung bình của G3 khi biến tăng một đơn vị, giữ các biến khác cố định.

Kết quả mong đợi: R² khoảng 0,304. Lưu ý đây là R² trên toàn bộ dữ liệu, không phải tập test.

Câu hỏi: viết một câu tiếng Việt giải thích hệ số của failures.

In [ ]:
X_const = sm.add_constant(X_A.astype(float))
ols = sm.OLS(y, X_const).fit()
print(ols.summary())

## Bước 10: Lọc các biến có ý nghĩa thống kê

Mục tiêu: lấy ra các biến có p-value dưới 0,05.

Kiến thức: ols.pvalues là một Series. Lọc bằng điều kiện ols.pvalues < 0.05, rồi bỏ hằng số bằng drop('const').

Kết quả mong đợi: 7 biến gồm studytime, failures, schoolsup, higher, internet, romantic, health. Hãy đối chiếu với bảng bạn vừa in.

Gợi ý thêm: thử fit OLS với feat_B (setup B) rồi xem biến nào mất ý nghĩa. Câu trả lời sẽ cho thấy điểm trước đó đã "hấp thụ" ảnh hưởng của những biến nào.

In [ ]:
sig = ols.pvalues[ols.pvalues < 0.05].drop('const')
print(sig.sort_values())

## Bước 11: Kiểm tra phần dư

Mục tiêu: vẽ phần dư (residual) để kiểm tra giả định của linear regression.

Kiến thức: phần dư bằng giá trị thực trừ giá trị dự đoán. Nếu mô hình ổn, các điểm nên phân tán ngẫu nhiên quanh đường 0 và không tạo hình dạng rõ ràng. Phần dư lớn bất thường thường đến từ các điểm ngoại lai.

Kết quả mong đợi: 15 học sinh có G3 = 0 đều có phần dư âm lớn, từ khoảng -5 đến -14.

Câu hỏi: nhóm điểm này có làm lệch đường hồi quy không? Bạn sẽ xử lý thế nào?

In [ ]:
plt.scatter(ols.fittedvalues, ols.resid, alpha=0.5)
plt.axhline(0, color='red')
plt.xlabel('Giá trị dự đoán')
plt.ylabel('Phần dư')
plt.title('Phần dư của OLS setup A')
plt.show()

print('Phần dư của học sinh G3 = 0:', ols.resid[y == 0].round(1).values)

## Bước 12: Cross-validation

Mục tiêu: đánh giá mô hình ổn định hơn một lần chia train/test.

Kiến thức: 5-fold chia dữ liệu thành 5 phần. Mỗi lần, một phần làm test và bốn phần còn lại làm train. Lấy trung bình của 5 lần.

Kết quả mong đợi: R² trung bình khoảng 0,126.

Câu hỏi: so với R² 0,304 ở bước 9, con số này nói lên điều gì về việc đánh giá mô hình trên dữ liệu đã dùng để huấn luyện?

In [ ]:
cv_r2 = cross_val_score(LinearRegression(), X_A, y, cv=5, scoring='r2')
print(cv_r2.round(3))
print('Trung bình:', round(cv_r2.mean(), 3))

## Bước 13: Logistic regression, chuẩn bị biến mục tiêu

Mục tiêu: đổi bài toán thành phân loại đậu hay trượt.

Kiến thức: (y >= 10).astype(int) cho ra 1 nếu đậu và 0 nếu trượt. Baseline accuracy là tỷ lệ của lớp lớn nhất. Một mô hình chỉ đoán lớp đó vẫn đạt đúng tỷ lệ này.

Kết quả mong đợi: tỷ lệ đậu khoảng 0,846.

Câu hỏi: nếu chỉ nhìn accuracy, một mô hình luôn đoán "đậu" đạt bao nhiêu? Điều này cho thấy gì về việc chọn chỉ số đánh giá?

In [ ]:
ypass = (y >= 10).astype(int)
print('Tỷ lệ đậu (baseline accuracy):', round(ypass.mean(), 3))

## Bước 14: Logistic regression, setup A

Mục tiêu: huấn luyện logistic regression trên setup A và đo accuracy và AUC.

Kiến thức: StandardScaler đưa các cột về cùng thang đo, điều này cần thiết cho logistic có regularization. make_pipeline gộp các bước lại. stratify=ypass giữ tỷ lệ đậu và trượt giống nhau ở hai tập. AUC đo khả năng xếp hạng: 0,5 là đoán ngẫu nhiên, 1 là hoàn hảo. Dùng predict_proba để lấy xác suất.

Kết quả mong đợi: accuracy khoảng 0,785, AUC khoảng 0,534.

Câu hỏi: accuracy này cao hơn hay thấp hơn baseline ở bước 13?

In [ ]:
Xtr_A, Xte_A, ptr, pte = train_test_split(X_A, ypass, test_size=0.2, random_state=42, stratify=ypass)
clf_A = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
clf_A.fit(Xtr_A, ptr)
print('Accuracy:', round(accuracy_score(pte, clf_A.predict(Xte_A)), 3))
print('AUC:', round(roc_auc_score(pte, clf_A.predict_proba(Xte_A)[:, 1]), 3))

## Bước 15: Viết hàm để chạy lại cho setup B

Mục tiêu: gói quy trình ở bước 14 vào một hàm, rồi dùng lại cho cả setup A và setup B. Viết hàm giúp tránh copy code và giảm lỗi.

Kiến thức: def tên_hàm(tham_số): định nghĩa hàm. Hàm có thể trả về giá trị bằng return.

Kết quả mong đợi: setup B có accuracy khoảng 0,892 và AUC khoảng 0,957.

Câu hỏi: vì sao AUC tăng mạnh khi thêm G1 và G2? Gợi ý: nhìn lại tương quan ở bước 4.

In [ ]:
def run_logistic(X, name):
    Xtr, Xte, ptr, pte = train_test_split(X, ypass, test_size=0.2, random_state=42, stratify=ypass)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    clf.fit(Xtr, ptr)
    acc = accuracy_score(pte, clf.predict(Xte))
    auc = roc_auc_score(pte, clf.predict_proba(Xte)[:, 1])
    print(name, '| accuracy:', round(acc, 3), '| AUC:', round(auc, 3))
    return clf

run_logistic(X_A, 'Setup A')
run_logistic(X_B, 'Setup B')

## Bước 16: Odds ratio

Mục tiêu: đổi hệ số log-odds thành odds ratio để dễ đọc.

Kiến thức: sm.Logit là logistic regression trong statsmodels, cho p-value. Odds ratio bằng exp(coef). OR lớn hơn 1 nghĩa là khi biến tăng thì khả năng đậu tăng.

Kết quả mong đợi: OR của G2 khoảng 5,08. OR của G1 khoảng 2,15. OR của age khoảng 1,85 với p-value khoảng 0,003.

Câu hỏi: age có OR lớn và có ý nghĩa thống kê. Bạn nghĩ có thể giải thích thế nào? Nếu không chắc, ghi rõ đó là giả thuyết cần kiểm tra thêm.

In [ ]:
logit = sm.Logit(ypass, sm.add_constant(X_B.astype(float))).fit(disp=0)
odds = pd.DataFrame({'OR': np.exp(logit.params), 'p-value': logit.pvalues})
print(odds.loc[['G1', 'G2', 'age']].round(3))

## Bước 17: Cây quyết định

Mục tiêu: huấn luyện một cây nhỏ và đọc quy tắc của nó.

Kiến thức: max_depth giới hạn số tầng của cây. export_text in cây dưới dạng văn bản. Đọc từ trên xuống, mỗi dòng là một điều kiện rẽ nhánh. Giá trị ở lá là dự đoán trung bình của các mẫu rơi vào lá đó.

Kết quả mong đợi: nút gốc chia theo G2 <= 11.5.

Câu hỏi: nếu G2 = 14, cây dự đoán G3 là bao nhiêu? Trả lời bằng cách đọc cây in ra, không tính tay.

In [ ]:
tree = DecisionTreeRegressor(max_depth=3, random_state=42)
tree.fit(X_train_B, y_train)
print(export_text(tree, feature_names=feat_B, decimals=1))

## Bước 18: Độ sâu cây và overfitting

Mục tiêu: thấy rõ cây quá sâu sẽ học thuộc dữ liệu.

Kiến thức: với mỗi giá trị max_depth, so sánh R² trên tập train và R² trên tập test. Khoảng cách lớn giữa hai con số là dấu hiệu overfitting.

Kết quả mong đợi: độ sâu 4 cho R² test khoảng 0,055. Độ sâu không giới hạn (None) cho R² test khoảng -0,805 và R² train gần 1.

Câu hỏi: bạn chọn độ sâu nào? Giải thích bằng số liệu bạn in ra.

In [ ]:
for depth in [2, 3, 4, 6, None]:
    t = DecisionTreeRegressor(max_depth=depth, random_state=42)
    t.fit(X_train_A, y_train)
    r2_train = t.score(X_train_A, y_train)
    r2_test = r2_score(y_test, t.predict(X_test_A))
    print('max_depth =', depth, '| R² train:', round(r2_train, 3), '| R² test:', round(r2_test, 3))

## Bước 19: Random forest và feature importance

Mục tiêu: huấn luyện random forest và xem mức độ quan trọng của từng feature.

Kiến thức: random forest gồm nhiều cây, mỗi cây học trên một mẫu bootstrap. feature_importances_ cho điểm quan trọng của từng feature, tổng bằng 1.

Kết quả mong đợi: R² test khoảng 0,189. failures đứng đầu với khoảng 0,210.

Câu hỏi: failures xuất hiện đầu cả ở OLS và random forest. Điều này làm kết luận đáng tin hơn hay chỉ là trùng hợp? Giải thích.

In [ ]:
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train_A, y_train)
print('R² test:', round(r2_score(y_test, rf.predict(X_test_A)), 3))

imp = pd.Series(rf.feature_importances_, index=feat_A).sort_values(ascending=False)
print(imp.head(8).round(3))

## Bước 20: Permutation importance

Mục tiêu: đo mức quan trọng bằng cách xáo trộn từng cột trên tập test rồi xem R² giảm bao nhiêu.

Kiến thức: nếu xáo trộn một cột làm mô hình kém đi nhiều, cột đó quan trọng. Cách này ít bị thiên vị hơn feature_importances_, vốn hay ưu tiên các cột có nhiều giá trị khác nhau.

Kết quả mong đợi: so sánh thứ hạng với bước 19.

Câu hỏi: có biến nào đổi thứ hạng đáng kể không? Nếu có, bạn đoán vì sao?

In [ ]:
perm = permutation_importance(rf, X_test_A, y_test, n_repeats=10, random_state=42)
perm_imp = pd.Series(perm.importances_mean, index=feat_A).sort_values(ascending=False)
print(perm_imp.head(8).round(3))

## Bước 21: Bảng so sánh các mô hình

Mục tiêu: ghép kết quả của ba mô hình vào một bảng để so sánh.

Kiến thức: dict lưu tên mô hình. Vòng for lặp qua từng setup. pd.DataFrame(rows) tạo bảng từ danh sách các dict.

Kết quả mong đợi: linear regression setup A có test R² khoảng 0,219 và CV R² khoảng 0,126. Hãy đối chiếu các con số còn lại với bảng so sánh trong phần giải thích.

Câu hỏi: ở setup B, mô hình nào tốt nhất? Mô hình đơn giản nhất có đủ tốt không?

In [ ]:
models = {
    'Linear regression': LinearRegression(),
    'Cây độ sâu 4': DecisionTreeRegressor(max_depth=4, random_state=42),
    'Random forest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
}
rows = []
for name, model in models.items():
    row = {'Mô hình': name}
    for setup, X_all, X_tr, X_te in [('A', X_A, X_train_A, X_test_A), ('B', X_B, X_train_B, X_test_B)]:
        model.fit(X_tr, y_train)
        row[f'{setup} test R2'] = round(r2_score(y_test, model.predict(X_te)), 3)
        row[f'{setup} CV R2'] = round(cross_val_score(model, X_all, y, cv=5, scoring='r2').mean(), 3)
    rows.append(row)
print(pd.DataFrame(rows).to_string())

## Bước 22: Tự thử nghiệm

Không có code mẫu ở bước này. Hãy tự chọn ít nhất hai việc sau:

1. Bỏ failures khỏi feat_A rồi chạy lại linear regression. R² test thay đổi bao nhiêu?
2. Thay LinearRegression bằng Ridge (from sklearn.linear_model import Ridge). So sánh kết quả.
3. Thử RandomForestRegressor với max_depth=5 và min_samples_leaf=5. Có tốt hơn không?
4. Viết ba câu tiếng Anh tóm tắt bảng so sánh ở bước 21. Đây là cách luyện viết báo cáo.